# Design Spaces 

## Setup

### Import Packages and Directories

In [1]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path

from flowtangent.models.kernels import RBFKernel
from flowtangent.models.surrogates import SVGPSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

In [2]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name:str):

    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(row:dict):
    af_name = row["foil_name"].item()
    row["y_af"] = surf_lookup(af_name)
    return row

### Import Dataset

In [7]:
data_dir = Path("/home/jordan/dev/datasets/XFOIL_4D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)

In [13]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="jtsmart/XFOIL-4D", repo_type="dataset", filename="XFOIL_4D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting XFOIL 4D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

Stored dataset detected.
Dataset ready at: /home/jordan/dev/datasets/XFOIL_4D.zarr


In [18]:
XFD = ft.utils.data.XArrayDataset(data_dir, engine="zarr").to_tabular(["run", "alpha"])
# XFD = XFD.dropna(["alpha"])
# XFD = XFD.transform_variable(name2surf)

In [19]:
XFD[0]

AttributeError: 'XArrayDataset' object has no attribute '_transforms'

In [5]:
unique_names = [str(n) for n in set(np.asarray(XFD.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Setup Manifold

In [7]:
airfold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
airfold = airfold.fit(X_SURF)

confold = Passthrough(dim=5) # Mach, Re, Flap Angle, Flap Loc., Alpha
mfold = airfold + confold

### Setup Surrogate

In [8]:
krnl = RBFKernel(input_dim=21)
surr = SVGPSurrogate(
    jax.random.key(42),
    input_dim=21,
    output_dim=1,
    kernel=krnl,
    num_inducing=1024
)

### Setup Parameters and Requirements

In [ ]:
# Inputs
y_af = Parameter(name="y_af")
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha)

CL_max = Requirement(name="CL", lower_bound=1.2)
alpha_range = Requirement(name="alpha", lower_bound=0.0, upper_bound=5.0)
reqs = (CL_max, alpha_range)

### Setup Design Space

In [11]:
airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
    requirements=reqs
)